# Protection Strategy Scoring and Revision

AI-generated synthesis routes frequently contain **selectivity issues** — reactions where multiple functional groups could react, leading to unintended side products. This tutorial demonstrates how to detect competing sites, score routes for selectivity quality, and apply deterministic post-search protection/deprotection revision using the canonical `synplan.routes.quality.protection` module.

The approach is inspired by:

> Westerlund, A. M.; Sigmund, L. M.; Kannas, C.; Genheden, S.; Kabeshov, M.  
> *"Toward lab-ready AI synthesis plans with protection strategies and route scoring."*  
> ChemRxiv, 2025. [doi:10.26434/chemrxiv-2025-gdrr8](https://doi.org/10.26434/chemrxiv-2025-gdrr8)

SynPlanner's revision path here is intentionally **chython-native** and deterministic: it does not load Chemformer or qmdesc, and it treats the bundled protecting-group CSV rows as fragment descriptors rather than RDKit reaction templates.

## What this tutorial covers

1. **Functional group detection**: identify reactive FGs in molecules via SMARTS
2. **Competing site identification**: find FGs that could interfere with a reaction
3. **Reaction type classification**: classify reactions using CGR bond analysis
4. **Route scanning**: walk a route step-by-step to find all competing interactions
5. **S(T) scoring**: compute the competing-sites score for routes
6. **Atom-mapped scan records**: preserve `fg_atoms`, `anchor_atom`, and `center_atoms`
7. **Route re-ranking**: combine S(T) with the search score for selectivity-aware ranking
8. **Route revision**: insert conservative protection, protected transformation, and deprotection steps


## 1. Setup

In [1]:
from pathlib import Path
from synplan.utils.loading import download_preset

# download SynPlanner preset data
paths = download_preset("synplanner-gps", save_to="synplan_data")
ranking_policy_network = paths["ranking_policy"]
reaction_rules_path = paths["reaction_rules"]
# use your custom building blocks if needed
building_blocks_path = paths["building_blocks"]

In [2]:
import synplan
from synplan.routes.quality.protection import (
    ProtectionConfig,
    ProtectionRevisionConfig,
    FunctionalGroupDetector,
    FunctionalGroupMatch,
    IncompatibilityMatrix,
    RouteScanner,
    CompetingSitesScore,
    ProtectionRouteReviser,
    classify_reaction_type,
    get_reaction_center_atoms,
)
print("SynPlanner routes.quality.protection module loaded successfully")

# Older synplan.route_quality imports remain compatibility aliases, but new code
# should use the canonical synplan.routes.quality namespace shown above.


SynPlanner routes.quality.protection module loaded successfully


## 2. Functional Group Detection

The `FunctionalGroupDetector` uses SMARTS patterns to identify reactive functional groups in molecules. The bundled library contains 19 patterns across three categories:

| Category | Groups |
|----------|--------|
| Nucleophile | hydroxyl, phenol, primary/secondary amine, thiol, carboxylic acid, pyrrole/indole/amide NH, sulfonamide |
| Electrophile | aldehyde, ketone, acyl chloride, epoxide, isocyanate |
| Unsaturated | alkene, alkyne, Michael acceptor, diene |

In [3]:
from chython import smiles

config = ProtectionConfig()
detector = FunctionalGroupDetector(config.competing_groups_path)

print(f"Loaded {len(detector._patterns)} SMARTS patterns")

Loaded 102 SMARTS patterns


### Detect FGs in a single molecule

Let's examine a molecule with multiple reactive functional groups — an amino acid derivative with hydroxyl, amine, and carboxylic acid groups.

In [4]:
# Serine: amino acid with OH, NH2, and COOH
mol = smiles("OC(C(N)C(O)=O)=O")
display(mol)

matches = detector.detect_all(mol)
print(f"Found {len(matches)} functional group matches:\n")
for m in matches:
    print(f"  {m.name:20s} ({m.category:14s})  atoms: {m.atom_indices}")

Found 4 functional group matches:

  NonProlineAlphaAminoAcid_unprotected (AminoAcid     )  atoms: (4,)
  Acid_SaturatedAliphatic (Acid          )  atoms: (6,)
  Acid_SaturatedAliphatic (Acid          )  atoms: (1,)
  Amine_Primary_SaturatedAliphatic (Amine         )  atoms: (4,)


### Identify competing FGs

Given a set of reaction center atoms, `detect_competing()` returns FGs outside the reaction center, which could interfere with the reaction.

In [5]:
# Suppose the reaction center involves the carboxylic acid (atoms at specific indices)
# First, let's see what atoms are in the molecule
for n, atom in mol.atoms():
    print(f"  atom {n}: {atom}")

print("\n--- Detecting competing FGs (excluding reaction center) ---")
# Pick the first two atoms as a hypothetical reaction center
first_match = matches[0]
reaction_center = set(first_match.atom_indices)
print(f"Reaction center atoms: {reaction_center} ({first_match.name})")

competing = detector.detect_competing(mol, reaction_center)
print(f"\nCompeting FGs ({len(competing)}):")
for c in competing:
    print(f"  {c.name:20s} ({c.category:14s})  atoms: {c.atom_indices}")

  atom 1: O()
  atom 2: C()
  atom 3: C()
  atom 4: N()
  atom 5: C()
  atom 6: O()
  atom 7: O()
  atom 8: O()

--- Detecting competing FGs (excluding reaction center) ---
Reaction center atoms: {4} (NonProlineAlphaAminoAcid_unprotected)

Competing FGs (2):
  Acid_SaturatedAliphatic (Acid          )  atoms: (6,)
  Acid_SaturatedAliphatic (Acid          )  atoms: (1,)


## 3. Reaction Classification

The reaction classifier uses CGR (Condensed Graph of Reaction) bond analysis to classify reactions into broad types:

- **bond_formation**: only new bonds are formed
- **bond_breaking**: only existing bonds are broken
- substitution: bonds are both formed and broken (most common)
- other: no detectable bond changes

In [6]:
from chython import smiles as parse_smiles

# Fischer esterification (mapped SMILES)
rxn = parse_smiles("[CH3:1][C:2](=[O:4])[OH:3].[CH3:5][CH2:6][OH:7]>>[CH3:1][C:2](=[O:4])[O:7][CH2:6][CH3:5].[OH2:3]")

center = get_reaction_center_atoms(rxn)
rxn_type = classify_reaction_type(rxn)

print(f"Reaction center atoms: {center}")
print(f"Reaction type: {rxn_type}")

Reaction center atoms: {2, 3, 7}
Reaction type: substitution


## 4. Run Retrosynthetic Planning

Let's plan routes for a target molecule and then score them for selectivity issues. We use capivasertib — an FDA-approved anti-cancer drug with multiple reactive functional groups.

**Integrated scoring:** By passing a `ProtectionRouteScorer` to the `Tree`, route scores are automatically weighted by S(T). This means `tree.route_score(node_id)` already reflects protection quality — no manual re-ranking needed.

**Post-search revision:** After extracting routes, `ProtectionRouteReviser` can use saved atom mappings from the scanner to insert conservative protection/deprotection steps without calling Chemformer or qmdesc.


In [7]:
from synplan.chem.utils import mol_from_smiles

# Capivasertib — multiple reactive FGs (amide, amine, hydroxyl, heterocycles)
target_smiles = "NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2nc[nH]c3nccc2-3)CC1"
target_mol = mol_from_smiles(target_smiles, clean2d=True, standardize=True, clean_stereo=True)
display(target_mol)

# Check what FGs the target has
target_fgs = detector.detect_all(target_mol)
print(f"Target molecule FGs ({len(target_fgs)}):")
for fg in target_fgs:
    print(f"  {fg.name} ({fg.category})")

Target molecule FGs (2):
  PrimaryAlcoholAliphatic (Alcohol)
  Amine_Primary_SaturatedAliphatic (Amine)


In [8]:
from synplan.mcts.tree import Tree
from synplan.routes.quality.scorer import ProtectionRouteScorer
from synplan.utils.config import TreeConfig, RolloutEvaluationConfig
from synplan.utils.loading import (
    load_building_blocks,
    load_reaction_rules,
    load_policy_function,
    load_evaluation_function,
)

# Load resources and run search
building_blocks = load_building_blocks(building_blocks_path, standardize=False)
reaction_rules = load_reaction_rules(reaction_rules_path)
policy_network = load_policy_function(weights_path=ranking_policy_network)

tree_config = TreeConfig(
    search_strategy="expansion_first",
    max_iterations=300,
    max_time=120,
    max_depth=9,
    min_mol_size=1,
    init_node_value=0.5,
    ucb_type="uct",
    c_ucb=0.1,
)

eval_config = RolloutEvaluationConfig(
    policy_network=policy_network,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks,
    min_mol_size=tree_config.min_mol_size,
    max_depth=tree_config.max_depth,
    normalize=True,
)
evaluation_function = load_evaluation_function(eval_config)

# Protection-aware scorer: route_score() will automatically apply S(T) weighting
route_scorer = ProtectionRouteScorer.from_config()

tree = Tree(
    target=target_mol,
    config=tree_config,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks,
    expansion_function=policy_network,
    evaluation_function=evaluation_function,
    route_scorer=route_scorer,
)

for solved, node_id in tree:
    pass

print(f"Found {len(tree.winning_nodes)} routes")
print(f"route_score() now includes S(T) protection weighting automatically")
tree

  0%|          | 0/300 [00:00<?, ?it/s]

Found 158 routes
route_score() now includes S(T) protection weighting automatically


Tree for: c1cc(ccc1Cl)C(NC(C2(CCN(CC2)c3c4cc[nH]c4ncn3)N)=O)CCO
Time: 44.5 seconds
Number of nodes: 8994
Number of iterations: 300
Number of visited nodes: 294
Number of found routes: 158

## 5. Extract and Score Routes

Extract routes as `routes_dict` (the standard format: `{route_id: {step_id: ReactionContainer}}`), then score each route with S(T).

In [9]:
from synplan.routes.route_cgr import extract_reactions

routes_dict = extract_reactions(tree)
print(f"Extracted {len(routes_dict)} routes")

# Show route lengths
for rid, route in list(routes_dict.items())[:5]:
    print(f"  Route {rid}: {len(route)} steps")

Extracted 158 routes
  Route 223: 3 steps
  Route 224: 3 steps
  Route 225: 3 steps
  Route 245: 3 steps
  Route 255: 4 steps


### Set up the protection scorer

The scorer pipeline is:
```
FunctionalGroupDetector -> RouteScanner -> CompetingSitesScore
         ^                      ^
  SMARTS patterns    IncompatibilityMatrix
```

In [10]:
matrix = IncompatibilityMatrix(config.incompatibility_path)
scanner = RouteScanner(detector, matrix)
scorer = CompetingSitesScore(scanner)

print("Scorer ready.")

Scorer ready.


### Score a single route in detail

Let's look at one route and see exactly which competing interactions were found.

In [11]:
first_rid = next(iter(routes_dict))
first_route = routes_dict[first_rid]

score, interactions = scorer.score_route(first_route)

print(f"Route {first_rid}: S(T) = {score:.3f}")
print(f"Number of steps: {len(first_route)}")
print(f"Number of competing interactions: {len(interactions)}")

if interactions:
    print()
    print("Detailed interactions:")
    for inter in interactions:
        print(
            f"  Step {inter.step_id}: "
            f"{inter.fg_name} vs {inter.reacting_fg or 'unknown'} -> {inter.severity} "
            f"| fg_atoms={inter.fg_atoms}, anchor={inter.anchor_atom}"
        )


Route 223: S(T) = 0.500
Number of steps: 3
Number of competing interactions: 8

Detailed interactions:
  Step 0: NonProlineAlphaAminoAcid_unprotected vs Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic -> competing | fg_atoms=(1,), anchor=1
  Step 0: Acid_SaturatedAliphatic vs Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic -> compatible | fg_atoms=(31,), anchor=31
  Step 0: Amine_Primary_SaturatedAliphatic vs Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic -> competing | fg_atoms=(1,), anchor=1
  Step 1: NonProlineAlphaAminoAcid_unprotected vs Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic -> competing | fg_atoms=(1,), anchor=1
  Step 1: Acid_SaturatedAliphatic vs Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic -> compatible | fg_atoms=(31,), anchor=31
  Step 1: Amine_Primary_SaturatedAliphatic vs Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic -> competing | fg_atoms=(1,), anchor=1
  Step 2: PrimaryAlcoholAliphat

### Atom-mapped scan records

`RouteScanner.scan_route(route, detailed=True)` returns the same interactions plus stable atom-map handles that can be used later for deterministic revision. This avoids re-selecting a competing site by SMARTS after the site has already been detected.

- `fg_atoms`: all atom-map ids in the competing functional group
- `anchor_atom`: the atom-map id where a protecting group can attach
- `center_atoms`: atom-map ids in the intended reaction center


In [12]:
detailed_scan = scanner.scan_route(first_route, detailed=True)

print(f"Processed steps: {detailed_scan.processed_steps}")
print(f"Skipped steps:   {detailed_scan.skipped_steps}")
print(f"Halogen count:   {detailed_scan.halogen_count}")
print()

for inter in detailed_scan.interactions[:10]:
    print(
        f"step={inter.step_id:2d}  severity={inter.severity:12s}  "
        f"fg={inter.fg_name}  reacting={inter.reacting_fg or 'unknown'}"
    )
    print(
        f"    fg_atoms={inter.fg_atoms}  anchor_atom={inter.anchor_atom}  "
        f"center_atoms={inter.center_atoms}"
    )


Processed steps: (0, 1, 2)
Skipped steps:   ()
Halogen count:   0

step= 0  severity=competing     fg=NonProlineAlphaAminoAcid_unprotected  reacting=Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic
    fg_atoms=(1,)  anchor_atom=1  center_atoms=(19, 34)
step= 0  severity=compatible    fg=Acid_SaturatedAliphatic  reacting=Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic
    fg_atoms=(31,)  anchor_atom=31  center_atoms=(19, 34)
step= 0  severity=competing     fg=Amine_Primary_SaturatedAliphatic  reacting=Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic
    fg_atoms=(1,)  anchor_atom=1  center_atoms=(19, 34)
step= 1  severity=competing     fg=NonProlineAlphaAminoAcid_unprotected  reacting=Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic
    fg_atoms=(1,)  anchor_atom=1  center_atoms=(19, 20, 32)
step= 1  severity=compatible    fg=Acid_SaturatedAliphatic  reacting=Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic
    fg_atoms=(31,)  anc

### Score all routes

Compute S(T) for every route and look at the distribution.

In [13]:
scores = {}
for rid, route in routes_dict.items():
    s, _ = scorer.score_route(route)
    scores[rid] = s

# Distribution summary
score_vals = list(scores.values())
n_perfect = sum(1 for s in score_vals if s == 1.0)
n_good = sum(1 for s in score_vals if 0.75 <= s < 1.0)
n_moderate = sum(1 for s in score_vals if 0.5 <= s < 0.75)
n_poor = sum(1 for s in score_vals if s < 0.5)

print(f"S(T) score distribution across {len(scores)} routes:")
print(f"  Perfect  (S=1.0):     {n_perfect:4d}  ({100*n_perfect/len(scores):.1f}%)")
print(f"  Good     (0.75-1.0):  {n_good:4d}  ({100*n_good/len(scores):.1f}%)")
print(f"  Moderate (0.5-0.75):  {n_moderate:4d}  ({100*n_moderate/len(scores):.1f}%)")
print(f"  Poor     (<0.5):      {n_poor:4d}  ({100*n_poor/len(scores):.1f}%)")
print(f"\nMean S(T): {sum(score_vals)/len(score_vals):.3f}")
print(f"Min  S(T): {min(score_vals):.3f}")
print(f"Max  S(T): {max(score_vals):.3f}")

S(T) score distribution across 158 routes:
  Perfect  (S=1.0):        0  (0.0%)
  Good     (0.75-1.0):   104  (65.8%)
  Moderate (0.5-0.75):    54  (34.2%)
  Poor     (<0.5):         0  (0.0%)

Mean S(T): 0.740
Min  S(T): 0.500
Max  S(T): 0.929


## 6. Route Re-ranking

Since we passed a `ProtectionRouteScorer` to the `Tree`, `tree.route_score(node_id)` already returns the protection-weighted score:

```
score = original * S(T)
```

The manual re-ranking below demonstrates the `rank_routes()` API for cases where you want a different blending (e.g. additive with custom weight), or when working with saved routes outside of the tree.

In [14]:

# Get original route scores from the tree
existing_scores = {
    node_id: tree.route_score(node_id)
    for node_id in tree.winning_nodes
}

# Re-rank with combined score
ranked = scorer.rank_routes(
    routes_dict,
    existing_scores=existing_scores,
    weight=0.75,
)

print(f"{'Rank':>4}  {'Route':>6}  {'Combined':>8}  {'S(T)':>6}  {'Original':>8}")
print("-" * 42)
for i, (rid, combined, protection, original) in enumerate(ranked[:15]):
    print(f"{i+1:4d}  {rid:6d}  {combined:8.3f}  {protection:6.3f}  {original:8.4f}")

Rank   Route  Combined    S(T)  Original
------------------------------------------
   1    1092     0.875   0.833    0.1392
   2    1097     0.875   0.833    0.1392
   3    1100     0.875   0.833    0.1392
   4    1106     0.875   0.833    0.1392
   5    1118     0.870   0.875    0.1189
   6    1140     0.857   0.900    0.1013
   7    1147     0.857   0.900    0.1013
   8    1148     0.857   0.900    0.1013
   9    3113     0.842   0.917    0.0860
  10    3115     0.842   0.917    0.0860
  11    3120     0.842   0.917    0.0860
  12    3130     0.824   0.929    0.0712
  13    8148     0.811   0.857    0.0934
  14    8151     0.811   0.857    0.0934
  15    8152     0.811   0.857    0.0934


### Compare original vs re-ranked top routes

See which routes moved up (fewer selectivity issues) or down (more issues).

In [15]:
# Original ranking (by search score)
original_ranking = sorted(existing_scores.items(), key=lambda x: x[1], reverse=True)
original_top10 = [rid for rid, _ in original_ranking[:10]]

# New ranking (by combined score)
new_top10 = [rid for rid, _, _, _ in ranked[:10]]

promoted = set(new_top10) - set(original_top10)
demoted = set(original_top10) - set(new_top10)

print(f"Original top-10: {original_top10}")
print(f"New top-10:      {new_top10}")
print(f"\nPromoted into top-10:  {promoted or 'none'}")
print(f"Demoted from top-10:   {demoted or 'none'}")

Original top-10: [1092, 1097, 1100, 1106, 1118, 8116, 1140, 1147, 1148, 8763]
New top-10:      [1092, 1097, 1100, 1106, 1118, 1140, 1147, 1148, 3113, 3115]

Promoted into top-10:  {3113, 3115}
Demoted from top-10:   {8763, 8116}


## 7. Deterministic Route Revision

The reviser consumes the detailed scan records, chooses conservative chython-valid protecting fragments, and inserts a protection step, protected transformation, and deprotection step around the affected reaction. The implementation uses stable atom mappings from detection rather than a model-based protecting-group classifier.


In [16]:
from synplan.routes.quality.protection import (
    ProtectionRevisionConfig,
    ProtectionRouteReviser,
)
from synplan.routes.route_cgr import compose_route_cgr

revision_config = ProtectionRevisionConfig(max_revisions_per_route=1)
reviser = ProtectionRouteReviser.from_config(revision_config)

problem_rid, problem_score = min(scores.items(), key=lambda item: item[1])
revision = reviser.revise_route(routes_dict[problem_rid], route_id=problem_rid)

print(f"Route {problem_rid}")
print(f"  original S(T): {revision.original_score:.3f}")
print(f"  revised  S(T): {revision.revised_score:.3f}")
print(f"  accepted:      {revision.accepted}")
print(f"  revised steps: {len(revision.route)}")

if revision.actions:
    print()
    print("Accepted actions:")
    for action in revision.actions:
        print(
            f"  step {action.original_step_id} -> {action.new_step_ids}: "
            f"{action.fg_name} atoms={action.fg_atoms} anchor={action.anchor_atom} "
            f"with rule={action.rule_name} ({action.protection_class})"
        )
else:
    print()
    print("No deterministic revision was accepted for this route.")

print()
print("Rejected candidate attempts / unsupported sites (first 10):")
for diagnostic in revision.diagnostics[:10]:
    site = f"step={diagnostic.step_id} fg={diagnostic.fg_name}"
    rule = f" rule={diagnostic.rule_name}" if diagnostic.rule_name else ""
    score = ""
    if diagnostic.candidate_score is not None:
        score = f" score={diagnostic.candidate_score:.3f}"
        if diagnostic.baseline_score is not None:
            score += f" baseline={diagnostic.baseline_score:.3f}"
    detail = f" detail={diagnostic.detail}" if diagnostic.detail else ""
    print(f"  {diagnostic.reason}: {site}{rule}{score}{detail}")
if not revision.diagnostics:
    print("  none")

route_cgr_check = compose_route_cgr({problem_rid: revision.route}, problem_rid)
route_cgr = route_cgr_check['cgr']
route_cgr.clean2d()
display(route_cgr)
display(route_cgr_check)
status = 'passed' if route_cgr_check is not None else 'failed'
print()
print(f"Route-CGR validation: {status}")


Route 223
  original S(T): 0.500
  revised  S(T): 0.800
  accepted:      True
  revised steps: 5

Accepted actions:
  step 0 -> (0, 1, 4): Amine_Primary_SaturatedAliphatic atoms=(1,) anchor=1 with rule=amine_nosyl (amine)

Rejected candidate attempts / unsupported sites (first 10):
  score_not_improved: step=0 fg=Amine_Primary_SaturatedAliphatic rule=amine_benzyl score=0.500 baseline=0.500 detail=candidate 0.500000 <= baseline 0.500000
  score_not_improved: step=0 fg=Amine_Primary_SaturatedAliphatic rule=amine_benzhydrylidene score=0.500 baseline=0.500 detail=candidate 0.500000 <= baseline 0.500000
  score_not_improved: step=0 fg=Amine_Primary_SaturatedAliphatic rule=amine_benzylidene score=0.500 baseline=0.500 detail=candidate 0.500000 <= baseline 0.500000
  score_not_improved: step=0 fg=Amine_Primary_SaturatedAliphatic rule=amine_chloro_tritil score=0.500 baseline=0.500 detail=candidate 0.500000 <= baseline 0.500000
  score_not_improved: step=0 fg=Amine_Primary_SaturatedAliphatic rul

{'cgr': <synplan.routes.route_cgr.container.RouteCGRContainer at 0x473714850>,
 'reactions_dict': {4: <chython.containers.reaction.ReactionContainer at 0x473227460>,
  3: <chython.containers.reaction.ReactionContainer at 0x4732275b0>,
  2: <chython.containers.reaction.ReactionContainer at 0x47357c280>,
  1: <chython.containers.reaction.ReactionContainer at 0x47357c670>,
  0: <chython.containers.reaction.ReactionContainer at 0x47357c7c0>}}


Route-CGR validation: passed


### Visualise the revised route

The revised route is not part of the original search tree, so it is rendered from the route JSON representation. The orange molecule box marks the protecting-group reagent that was added by the reviser.


In [18]:
from IPython.display import HTML, SVG, display
from synplan.routes.io import make_json
from synplan.utils.visualisation import get_route_svg_json

if revision.accepted:
    display(SVG(get_route_svg_json(
        make_json({problem_rid: routes_dict[problem_rid]}),
        problem_rid,
        labeled=True,
    )))

    revised_routes_json = make_json(
        {problem_rid: revision.route},
        route_metadata={problem_rid: revision.route_metadata},
    )
    revised_svg = get_route_svg_json(
        revised_routes_json,
        problem_rid,
        labeled=True,
    )
    display(
        HTML(
            f"<h4>Revised route {problem_rid} — "
            f"S(T): {revision.original_score:.3f} → "
            f"{revision.revised_score:.3f}</h4>"
        )
    )
    display(SVG(revised_svg))
else:
    revised_routes_json = {}
    revised_svg = None
    print("No accepted revision is available to visualise for this route.")


## Route Visualisation

Let's visualise routes to see how competing functional groups correlate with route structure. We'll compare:
1. A **problematic route** (lowest S(T)) that has many competing interactions
2. The **best available route** (highest S(T)) with the fewest selectivity issues

In [17]:
from IPython.display import SVG, display, HTML
from synplan.utils.visualisation import get_route_svg

# Sort routes by S(T) score
sorted_by_score = sorted(scores.items(), key=lambda x: x[1])

# Worst route (lowest S(T))
worst_rid, worst_score = sorted_by_score[0]
# Best route (highest S(T))
best_rid, best_score = sorted_by_score[-1]

print(f"Worst route: {worst_rid}  S(T) = {worst_score:.3f}")
print(f"Best route:  {best_rid}  S(T) = {best_score:.3f}")

Worst route: 223  S(T) = 0.500
Best route:  3130  S(T) = 0.929


### Problematic route (lowest S(T))

This route has the most competing functional group interactions. The detailed interaction table below the diagram shows which FGs conflict at each step.

In [18]:
# Visualise the worst route
worst_svg = get_route_svg(tree, worst_rid)
if worst_svg:
    display(HTML(f"<h4>Route {worst_rid} — S(T) = {worst_score:.3f}</h4>"))
    display(SVG(worst_svg))
else:
    print(f"Route {worst_rid} could not be visualised (not in winning nodes).")

In [19]:
# Show competing interactions for the worst route
worst_route = routes_dict[worst_rid]
_, worst_interactions = scorer.score_route(worst_route)
i_count, c_count, h_count = RouteScanner.classify_interactions(worst_interactions)

print(f"Route {worst_rid}: {len(worst_route)} steps, "
      f"{len(worst_interactions)} interactions "
      f"(I={i_count}, C={c_count}, H={h_count})\n")

# Group by step
from collections import defaultdict
by_step = defaultdict(list)
for inter in worst_interactions:
    by_step[inter.step_id].append(inter)

for step_id in sorted(by_step):
    step_inters = by_step[step_id]
    non_compatible = [i for i in step_inters if i.severity != "compatible"]
    reacting = step_inters[0].reacting_fg or "unknown"
    if non_compatible:
        print(f"Step {step_id} (reacting FG: {reacting}):")
        for i in non_compatible:
            marker = "!!" if i.severity == "incompatible" else " ~"
            print(f"  {marker} {i.fg_name} → {i.severity}")
        print()

Route 223: 3 steps, 8 interactions (I=0, C=5, H=0)

Step 0 (reacting FG: Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic):
   ~ NonProlineAlphaAminoAcid_unprotected → competing
   ~ Amine_Primary_SaturatedAliphatic → competing

Step 1 (reacting FG: Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic):
   ~ NonProlineAlphaAminoAcid_unprotected → competing
   ~ Amine_Primary_SaturatedAliphatic → competing

Step 2 (reacting FG: Acid_SaturatedAliphatic):
   ~ PrimaryAlcoholAliphatic → competing



### Best available route (highest S(T))

This route has the fewest selectivity concerns. In a real workflow, this is the route you'd prefer to execute in the lab.

In [19]:
# Visualise the best route
best_svg = get_route_svg(tree, best_rid)
if best_svg:
    display(HTML(f"<h4>Route {best_rid} — S(T) = {best_score:.3f}</h4>"))
    display(SVG(best_svg))
else:
    print(f"Route {best_rid} could not be visualised (not in winning nodes).")

In [20]:
# Show interactions for the best route
best_route = routes_dict[best_rid]
_, best_interactions = scorer.score_route(best_route)
i_best, c_best, h_best = RouteScanner.classify_interactions(best_interactions)

print(f"Route {best_rid}: {len(best_route)} steps, "
      f"{len(best_interactions)} interactions "
      f"(I={i_best}, C={c_best}, H={h_best})\n")

by_step_best = defaultdict(list)
for inter in best_interactions:
    by_step_best[inter.step_id].append(inter)

for step_id in sorted(by_step_best):
    step_inters = by_step_best[step_id]
    non_compatible = [i for i in step_inters if i.severity != "compatible"]
    reacting = step_inters[0].reacting_fg or "unknown"
    if non_compatible:
        print(f"Step {step_id} (reacting FG: {reacting}):")
        for i in non_compatible:
            marker = "!!" if i.severity == "incompatible" else " ~"
            print(f"  {marker} {i.fg_name} → {i.severity}")
        print()
    else:
        print(f"Step {step_id} (reacting FG: {reacting}): all compatible")

Route 2935: 6 steps, 7 interactions (I=0, C=0, H=0)

Step 0 (reacting FG: Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic): all compatible
Step 1 (reacting FG: Amine_CyclicSecondary_SaturatedAliphatic-SaturatedAliphatic): all compatible
Step 4 (reacting FG: Acid_SaturatedAliphatic): all compatible
Step 5 (reacting FG: unknown): all compatible


### Top re-ranked route

After combining the search score with the protection score, the top-ranked route represents the best trade-off between synthetic feasibility (original score) and selectivity (S(T)).

In [21]:
# Top re-ranked route (best combined score)
top_rid, top_combined, top_protection, top_original = ranked[0]

print(f"Top re-ranked route: {top_rid}")
print(f"  Combined score:   {top_combined:.3f}")
print(f"  Protection S(T):  {top_protection:.3f}")
print(f"  Original score:   {top_original:.4f}\n")

top_svg = get_route_svg(tree, top_rid)
if top_svg:
    display(HTML(f"<h4>Top re-ranked: Route {top_rid} — combined = {top_combined:.3f}</h4>"))
    display(SVG(top_svg))
else:
    print(f"Route {top_rid} could not be visualised.")

Top re-ranked route: 1299
  Combined score:   0.911
  Protection S(T):  1.000
  Original score:   0.1088



## 8. Standalone Usage (from saved routes)

You can also score and revise previously saved routes without re-running the search.


In [22]:
from synplan.routes.io import read_routes_json, write_routes_json
from synplan.routes.quality.protection import (
    ProtectionRevisionConfig,
    ProtectionRouteReviser,
)

# Example: load from a JSON file produced by a previous search
# routes_dict = read_routes_json("path/to/routes.json", to_dict=True)

# Set up scorer and reviser from scratch (uses bundled defaults)
config = ProtectionRevisionConfig()
reviser = ProtectionRouteReviser.from_config(config)
scanner = reviser.scanner
scorer = reviser.scorer

# Score and rank (protection score only, no original search score)
ranked_protection_only = scorer.rank_routes(routes_dict, weight=1.0)

print(f"{'Rank':>4}  {'Route':>6}  {'S(T)':>6}")
print("-" * 20)
for i, (rid, combined, protection, _) in enumerate(ranked_protection_only[:10]):
    print(f"{i+1:4d}  {rid:6d}  {protection:6.3f}")

# Revise all routes deterministically and export metadata with the route JSON
revised_routes = reviser.revise_routes(routes_dict)
revised_routes_dict = {rid: result.route for rid, result in revised_routes.items()}
revision_metadata = {rid: result.route_metadata for rid, result in revised_routes.items()}

# write_routes_json(revised_routes_dict, "revised_routes.json", route_metadata=revision_metadata)


Rank   Route    S(T)
--------------------
   1    1299   1.000
   2    1300   1.000
   3    1302   1.000
   4    1465   1.000
   5    1484   1.000
   6    1485   1.000
   7    1487   1.000
   8    2035   1.000
   9    2045   1.000
  10    2047   1.000


## 9. Revision Caveats

- The bundled `protection_group_templates.csv` rows came from an RDKit-oriented workflow. SynPlanner treats them as protecting-fragment descriptors, not executable reaction templates.
- The reviser is chython-native and validates edited molecules with SynPlanner/chython utilities.
- v1 route revision supports amine and hydroxyl/phenol single-anchor protections plus carbonyl-acetal protections.
- Carboxyl, thiol, and diol CSV rows are loaded as metadata-only data and are not revision candidates yet.
- Orthogonal protecting-group strategy and identical-site merging are intentionally left for later work.


## 10. Interpreting the Score

The competing-sites score S(T) is computed as:

$$S(\mathcal{T}) = \max\left[1 - \frac{I + 0.5 C + H}{\max(N_{\text{reactions}}, 1)},\; 0\right]$$

Where:
- **I** = number of highly **incompatible** functional group interactions (penalty = 1.0 each)
- **C** = number of **competing** interactions (penalty = 0.5 each)
- **H** = number of **halogen** competing sites (penalty = 1.0 each; reserved for future)
- **N** = number of reaction steps in the route

<div class="alert alert-info">
<b>Implementation note</b>

The actual implementation uses a <b>worst-per-step</b> variant of the formula above. Instead of summing all individual interaction penalties, each step contributes only the penalty of its most severe interaction (1.0 for incompatible, 0.5 for competing, 0.0 for compatible). This prevents drug-like molecules with many functional groups from overwhelming the score. The formula becomes: <code>S(T) = max[1 - (sum(w_s) + H) / max(N, 1), 0]</code>, where <code>w_s = max severity penalty among interactions in step s</code>.
</div>

| S(T) | Interpretation |
|------|---------------|
| 1.0 | No competing sites: route is clean |
| 0.75–0.99 | Minor issues (e.g., 1 competing FG in a 4-step route) |
| 0.5–0.75 | Moderate selectivity issues: may benefit from protecting groups |
| < 0.5 | Significant problems — route should be deprioritized |
| 0.0 | Maximally conflicting |

## References

- Westerlund, A. M. et al. "Toward lab-ready AI synthesis plans with protection strategies and route scoring." *ChemRxiv*, 2025. [doi:10.26434/chemrxiv-2025-gdrr8](https://doi.org/10.26434/chemrxiv-2025-gdrr8)
- ReactionUtils protection documentation: [molecularai.github.io/reaction_utils/rxnutils.chem.protection.html](https://molecularai.github.io/reaction_utils/rxnutils.chem.protection.html)
- ReactionUtils amino-acid protection implementation: [github.com/MolecularAI/reaction_utils/blob/main/rxnutils/chem/protection/amino_acids.py](https://github.com/MolecularAI/reaction_utils/blob/main/rxnutils/chem/protection/amino_acids.py)
- Kogej, T. et al. "SMARTS-RX: A SMARTS-Based Representation of Chemical Functions for Reactivity Analysis." *ChemRxiv*, 2025.
- SynPlanner documentation: [synplanner.readthedocs.io](https://synplanner.readthedocs.io/)


In [52]:
from chython import smiles, depict_settings
from synplan.routes.quality.protection import (
    ProtectionRevisionConfig,
    ProtectionRouteReviser,
)
from synplan.routes.route_cgr import compose_route_cgr

# rxn_smi = (
#     "[CH3:20][CH2:21][O:22][C:1](=[O:2])[CH:3]1[CH2:4][CH2:5]"
#     "[C:6](=[O:7])[CH2:8][CH2:9]1>>"
#     "[OH:2][CH2:1][CH:3]1[CH2:4][CH2:5][C:6](=[O:7])[CH2:8][CH2:9]1"
# )
# reaction = smiles(rxn_smi)
# reaction.clean2d()
# display(reaction)


# route2 = {0: reaction}

depict_settings(aam=False)

raw_route = {
    0: smiles(
        "[O:8]=[C:1]1[CH2:2][CH2:3][C:4](=[O:5])[CH2:6][CH2:7]1>>"
        "[OH:8][CH:1]1[CH2:2][CH2:3][C:4](=[O:5])[CH2:6][CH2:7]1"
    ),
    1: smiles(
        "[OH:8][CH:1]1[CH2:2][CH2:3][C:4](=[O:5])[CH2:6][CH2:7]1>>"
        "[CH:1]1[CH2:2][CH2:3][C:4](=[O:5])[CH2:6][CH:7]=1.[OH2:8]"
    ),
}
route = compose_route_cgr({0: raw_route}, 0)['reactions_dict']
routes_dict= {0: route}
problem_rid = 0
display(SVG(get_route_svg_json(
        make_json({problem_rid: routes_dict[problem_rid]}),
        problem_rid,
        labeled=True,
    )))

reviser = ProtectionRouteReviser.from_config(
    ProtectionRevisionConfig(max_revisions_per_route=1)
)

scan = reviser.scanner.scan_route(route, detailed=True)
score, _ = reviser.scorer.score_route(route)
revision = reviser.revise_route(route, route_id=0)

print("S(T):", score)
print("interactions:")
for item in scan.interactions:
    print(
        item.step_id,
        item.fg_name,
        item.fg_atoms,
        "anchor=", item.anchor_atom,
        "reacting=", item.reacting_fg,
        "severity=", item.severity,
        "center=", item.center_atoms,
    )

print("revision accepted:", revision.accepted)
print("revised S(T):", revision.revised_score)
print("actions:", revision.actions)
print("diagnostics:", revision.diagnostics[:10])

print("route-CGR valid:", compose_route_cgr({0: revision.route}, 0) is not None)

S(T): 0.75
interactions:
0 KetoneAliphaticCyclic (4,) anchor= 4 reacting= KetoneAliphaticCyclic severity= competing center= (1, 8)
1 KetoneAliphaticCyclic (4,) anchor= 4 reacting= SecondaryAlcoholAliphaticCyclic severity= compatible center= (1, 7, 8)
revision accepted: True
revised S(T): 0.875
actions: [ProtectionAction(original_step_id=0, new_step_ids=(0, 1, 3), fg_name='KetoneAliphaticCyclic', fg_atoms=(4,), anchor_atom=4, severity='competing', rule_name='carbonyl_dioxolane', strategy='carbonyl_acetal', p_mol='OCCO', protection_class='carbonyl')]
diagnostics: []
route-CGR valid: True


In [53]:
routes_dict= {0: route}
problem_rid = 0

In [54]:
from IPython.display import HTML, SVG, display
from synplan.routes.io import make_json
from synplan.utils.visualisation import get_route_svg_json

if revision.accepted:
    display(SVG(get_route_svg_json(
        make_json({problem_rid: routes_dict[problem_rid]}),
        problem_rid,
        labeled=True,
    )))

    revised_routes_json = make_json(
        {problem_rid: revision.route},
        route_metadata={problem_rid: revision.route_metadata},
    )
    revised_svg = get_route_svg_json(
        revised_routes_json,
        problem_rid,
        labeled=True,
    )
    display(
        HTML(
            f"<h4>Revised route {problem_rid} — "
            f"S(T): {revision.original_score:.3f} → "
            f"{revision.revised_score:.3f}</h4>"
        )
    )
    display(SVG(revised_svg))
else:
    revised_routes_json = {}
    revised_svg = None
    print("No accepted revision is available to visualise for this route.")


In [55]:
smarts('[N;D2:1]=;!@[C;D3;z2;x1](-[C;a;r6]:1:[C;D2]:[C;D2]:[C;D2]:[C;D2]:[C;D2]:1)-[C;a;r6]:2:[C;D2]:[C;D2]:[C;D2]:[C;D2]:[C;D2]:2')

In [57]:
l = smarts('[O;D1;$(O[C;D2,D3;$(C(=O));!$(C(O)(=O)[!#6])]);$([!$(**[C]=[C]);!$(**[C]#[C]);!$(**[C]#[N]);!$(**a)])]')
l.clean2d()
l